# artifacts: ошибки старого baseline

Восстанавливаем предсказания ResNet18 + CatBoost на прежней validation. Новую модель здесь не обучаем.

## 1. Данные и прежнее разбиение

In [ ]:
%pip install -q catboost iterative-stratification

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from catboost import CatBoostClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet18
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/train.csv').exists()), None)
if ROOT is None:
    ROOT = Path('/content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main')
import sys
sys.path.insert(0, str(ROOT))
from src.evaluation import binary_metrics, best_threshold
from src.data import split_indices, load_embeddings
from src.plots import setup_plot_style
setup_plot_style()
assert (ROOT / 'data/train.csv').exists(), ROOT

labels = pd.read_csv(ROOT / 'data/train.csv')
targets = labels.columns.drop('item_id').tolist()
split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=.15, random_state=42)
train_val_idx, test_idx = next(split.split(labels, labels[targets]))
train_val = labels.iloc[train_val_idx].reset_index(drop=True)
split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=.1765, random_state=43)
_, val_idx = next(split.split(train_val, train_val[targets]))
val = train_val.iloc[val_idx].reset_index(drop=True)
assert (len(val), len(test_idx)) == (1361, 1344)
print('Validation:', len(val), '| artifacts:', val['artifacts'].sum())

## 2. Вероятности ResNet18

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 336)), transforms.ToTensor(),
    transforms.Normalize([.485, .456, .406], [.229, .224, .225]),
])

class Renders(Dataset):
    def __len__(self):
        return len(val)

    def __getitem__(self, index):
        path = ROOT / 'data/train' / f'{val.item_id.iloc[index]}.png'
        with Image.open(path) as image:
            return transform(image.convert('RGB'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(targets))
checkpoint = torch.load(ROOT / 'archive/models/resnet/best_resnet18_improved.pth',
                        map_location='cpu', weights_only=False)
assert checkpoint['target_columns'] == targets
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device).eval()

resnet_probabilities = []
with torch.inference_mode():
    for images in tqdm(DataLoader(Renders(), batch_size=32, num_workers=0), desc='ResNet18'):
        p = model(images.to(device)).sigmoid().cpu().numpy()
        resnet_probabilities.append(p[:, targets.index('artifacts')])
resnet_p = np.concatenate(resnet_probabilities)
print('Готово:', len(resnet_p))

## 3. CatBoost и старая связка

In [ ]:
config = json.loads((ROOT / 'archive/artifacts/catboost_features_config.json').read_text(encoding='utf-8'))
features = pd.read_csv(ROOT / 'archive/artifacts/train_mesh_features_relative.csv').set_index('item_id')
catboost = CatBoostClassifier()
catboost.load_model(str(ROOT / 'archive/models/catboost_features/artifacts.cbm'))
catboost_p = catboost.predict_proba(features.loc[val.item_id, config['feature_columns']])[:, 1]
fusion_p = .5 * resnet_p + .5 * catboost_p
threshold = json.loads((ROOT / 'archive/artifacts/resnet_catboost_features_config.json').read_text(encoding='utf-8'))['thresholds']['artifacts']
prediction = fusion_p >= threshold
y = val['artifacts'].to_numpy()
tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
print('Precision:', round(precision_score(y, prediction), 4))
print('Recall:', round(recall_score(y, prediction), 4))
print('F1:', round(f1_score(y, prediction), 4), '| AP:', round(average_precision_score(y, fusion_p), 4))
print('FP:', fp, '| FN:', fn, '| Порог:', round(threshold, 2))
assert abs(f1_score(y, prediction) - .3153) < .001

errors = val[['item_id', 'artifacts', 'noisy', 'intersection', 'lowpoly', 'set']].copy()
errors['probability'] = fusion_p
errors['prediction'] = prediction.astype(int)
print('FP с noisy:', int(((errors.artifacts == 0) & (errors.prediction == 1) & (errors.noisy == 1)).sum()))
out = ROOT / 'labels/artifacts/artifacts'
out.mkdir(parents=True, exist_ok=True)
errors.to_csv(out / 'baseline_validation.csv', index=False)

## 4. Ложные срабатывания и пропуски

In [ ]:
def show_cases(cases):
    fig, axes = plt.subplots(1, len(cases), figsize=(25, 4))
    for ax, (_, row) in zip(axes, cases.iterrows()):
        with Image.open(ROOT / 'data/train' / f'{row.item_id}.png') as image:
            ax.imshow(image)
        ax.set_title(f'{row.item_id[:8]} | p={row.probability:.2f}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    display(cases.round(3))

fp_cases = errors[(errors.artifacts == 0) & (errors.prediction == 1)].nlargest(5, 'probability')
fn_cases = errors[(errors.artifacts == 1) & (errors.prediction == 0)].nsmallest(5, 'probability')
print('Ложные срабатывания:')
show_cases(fp_cases)
print('Пропущенные artifacts:')
show_cases(fn_cases)

## Вывод

Старая связка получила F1 = 0.3153, precision = 0.2319 и recall = 0.4923. Среди 106 FP у 70 объектов есть метка `noisy`: модель часто путает шумные сканы с артефактами. Среди просмотренных FN есть разошедшиеся части объекта и повреждённый край. Поэтому дальше проверим локальные геометрические признаки и связные компоненты. Шипы, лишние фрагменты, деформированные грани и локальные искажения пока остаются гипотезами: отдельных меток для этих подтипов в CSV нет.

## Вывод

Сравниваем на прежней validation. Test не используем для выбора порогов. Небольшой прирост после многих опытов ещё нужно перепроверять.